# LLaVA Pruned Inference

Use the saved layer-10 debiased image-token importances to prune LLaVA-1.5 image tokens before generation.

In [ ]:
%pip install --user -q -U transformers accelerate datasets pillow tqdm

In [ ]:
from pathlib import Path
import json

import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

In [ ]:
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
    if HERE.name != "llava" and (HERE / "llava").exists():
        HERE = HERE / "llava"

MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
MODEL_TAG = MODEL_NAME.rsplit("/", 1)[-1].replace("-", "_")
SAMPLES_PATH = HERE / "vqav2_samples" / "samples.jsonl"
TARGET_LAYER = 10
IMPORTANCE_DIR = HERE / f"{MODEL_TAG}_layer{TARGET_LAYER}_debiased_outputs"
OUTPUT_DIR = HERE / f"{MODEL_TAG}_pruned_outputs"

KEEP_RATIO = 0.5
MAX_NEW_TOKENS = 32
SAMPLE_INDEX = 0

OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def load_samples():
    if not SAMPLES_PATH.exists():
        raise FileNotFoundError(f"Missing {SAMPLES_PATH}")

    samples = []
    with open(SAMPLES_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            sample = json.loads(line)
            sample["sample_id"] = int(sample["sample_id"])
            sample["question_id"] = int(sample["question_id"])
            sample["image_id"] = int(sample["image_id"])

            image_path = Path(sample["image_path"])
            if not image_path.is_absolute():
                image_path = HERE / image_path
            sample["image_path"] = image_path
            samples.append(sample)

    print(f"Loaded {len(samples)} samples")
    return samples


samples = load_samples()

In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer
model_device = next(model.parameters()).device

print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

In [ ]:
def build_prompt(question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    try:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"USER: <image>\n{question}\nASSISTANT:"


def move_inputs(inputs):
    moved = {}
    for key, value in inputs.items():
        if not hasattr(value, "to"):
            moved[key] = value
        elif torch.is_floating_point(value):
            moved[key] = value.to(device=model_device, dtype=model.dtype)
        else:
            moved[key] = value.to(model_device)
    return moved


def prepare_inputs(sample):
    image = Image.open(sample["image_path"]).convert("RGB")
    prompt = build_prompt(sample["question"])
    inputs = processor(text=prompt, images=image, return_tensors="pt")
    return move_inputs(inputs)


def image_token_id():
    token_id = getattr(model.config, "image_token_index", None)
    if token_id is None:
        token_id = getattr(model.config, "image_token_id", None)
    if token_id is None:
        token_id = tokenizer.convert_tokens_to_ids("<image>")
    if token_id is None or token_id < 0:
        raise ValueError("Could not resolve LLaVA image token id")
    return int(token_id)


IMAGE_TOKEN_ID = image_token_id()

In [ ]:
def load_vision_importance(sample_id):
    path = IMPORTANCE_DIR / f"sample_{sample_id}.pt"
    if not path.exists():
        raise FileNotFoundError(f"Missing importance bundle {path}")

    bundle = torch.load(path, map_location="cpu")
    if bundle.get("model_name") != MODEL_NAME:
        raise ValueError(f"{path} was generated with {bundle.get('model_name')}, expected {MODEL_NAME}")
    if bundle.get("target_layer") != TARGET_LAYER:
        raise ValueError(f"{path} has target_layer={bundle.get('target_layer')}, expected {TARGET_LAYER}")
    if "debiased_mean" in bundle:
        importance = bundle["debiased_mean"]
    else:
        importance = bundle["debiased"].mean(0)

    return importance.float(), bundle


def llava_module(name):
    if hasattr(model, name):
        return getattr(model, name)
    if hasattr(model, "model") and hasattr(model.model, name):
        return getattr(model.model, name)
    raise AttributeError(f"Could not find LLaVA module {name}")


def get_image_features(inputs):
    pixel_values = inputs["pixel_values"].to(device=model_device, dtype=model.dtype)
    vision_tower = llava_module("vision_tower")
    projector = llava_module("multi_modal_projector")

    vision_feature_layer = getattr(model.config, "vision_feature_layer", -2)
    vision_feature_select_strategy = getattr(model.config, "vision_feature_select_strategy", "default")

    with torch.no_grad():
        vision_outputs = vision_tower(pixel_values, output_hidden_states=True, return_dict=True)

    if vision_outputs.hidden_states is None:
        raise ValueError("LLaVA vision tower did not return hidden states")

    features = vision_outputs.hidden_states[vision_feature_layer]
    if vision_feature_select_strategy == "default":
        features = features[:, 1:]
    elif vision_feature_select_strategy != "full":
        raise ValueError(f"Unsupported vision_feature_select_strategy={vision_feature_select_strategy!r}")

    return projector(features).to(model_device)


In [ ]:
def text_embedding_layer():
    if hasattr(model, "get_input_embeddings"):
        embeddings = model.get_input_embeddings()
        if embeddings is not None:
            return embeddings
    return model.language_model.get_input_embeddings()


def build_pruned_inputs(inputs, vision_importance, keep_ratio=KEEP_RATIO):
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    image_positions = torch.where(input_ids[0] == IMAGE_TOKEN_ID)[0]
    image_features = get_image_features(inputs)

    if image_features.ndim == 2:
        image_features = image_features.unsqueeze(0)
    if image_features.shape[1] != image_positions.numel():
        raise ValueError(f"image_features={image_features.shape[1]} image_positions={image_positions.numel()}")
    if vision_importance.numel() != image_positions.numel():
        raise ValueError(f"importance={vision_importance.numel()} image_positions={image_positions.numel()}")

    k = max(1, round(image_positions.numel() * keep_ratio))
    keep_idx = torch.sort(torch.topk(vision_importance.to(model_device), k).indices).values

    keep_seq = torch.ones_like(input_ids[0], dtype=torch.bool, device=model_device)
    keep_seq[image_positions] = False
    keep_seq[image_positions[keep_idx]] = True

    text_embeds = text_embedding_layer()(input_ids)
    new_ids = input_ids[:, keep_seq]
    new_embeds = text_embeds[:, keep_seq].clone()
    new_embeds[0, new_ids[0] == IMAGE_TOKEN_ID] = image_features[0, keep_idx].to(new_embeds.dtype)

    return {
        "input_ids": new_ids,
        "inputs_embeds": new_embeds,
        "attention_mask": attention_mask[:, keep_seq],
        "keep_idx": keep_idx.detach().cpu(),
        "num_image_tokens": int(image_positions.numel()),
        "num_kept_image_tokens": int(k),
    }


def run_unpruned(inputs, max_new_tokens=MAX_NEW_TOKENS):
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer_ids = ids[0, inputs["input_ids"].shape[1] :]
    return processor.decode(answer_ids, skip_special_tokens=True).strip()


def run_pruned(inputs, vision_importance, keep_ratio=KEEP_RATIO, max_new_tokens=MAX_NEW_TOKENS):
    pruned = build_pruned_inputs(inputs, vision_importance, keep_ratio)
    with torch.no_grad():
        ids = model.generate(
            inputs_embeds=pruned["inputs_embeds"],
            attention_mask=pruned["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    answer = processor.decode(ids[0], skip_special_tokens=True).strip()
    return ids, answer, pruned

In [ ]:
sample = samples[SAMPLE_INDEX]
importance, importance_bundle = load_vision_importance(sample["sample_id"])
inputs = prepare_inputs(sample)

unpruned_answer = run_unpruned(inputs)
generated_ids, pruned_answer, pruned = run_pruned(inputs, importance)

print("sample_id:", sample["sample_id"])
print("question:", sample["question"])
print("ground truth:", sample.get("ground_truth_answer"))
print("saved unpruned answer:", importance_bundle.get("answer"))
print("fresh unpruned answer:", unpruned_answer)
print("pruned answer:", pruned_answer)
print("kept image tokens:", pruned["num_kept_image_tokens"], "/", pruned["num_image_tokens"])
print("pruned input shape:", tuple(pruned["inputs_embeds"].shape))

In [ ]:
results = []
for sample in samples:
    importance, importance_bundle = load_vision_importance(sample["sample_id"])
    inputs = prepare_inputs(sample)
    _, pruned_answer, pruned = run_pruned(inputs, importance)
    row = {
        "sample_id": sample["sample_id"],
        "question_id": sample["question_id"],
        "image_id": sample["image_id"],
        "question": sample["question"],
        "ground_truth_answer": sample.get("ground_truth_answer"),
        "unpruned_answer": importance_bundle.get("answer"),
        "pruned_answer": pruned_answer,
        "keep_ratio": KEEP_RATIO,
        "num_image_tokens": pruned["num_image_tokens"],
        "num_kept_image_tokens": pruned["num_kept_image_tokens"],
    }
    results.append(row)
    print(row["sample_id"], "kept", row["num_kept_image_tokens"], "/", row["num_image_tokens"], "->", pruned_answer)

output_path = OUTPUT_DIR / f"pruned_results_keep_{KEEP_RATIO:.2f}.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=True) + "\n")

print("wrote", output_path)
